# Compare two robot dataset audits

Two public cube-manipulation datasets, [`triton7777/cube_to_bowl_so100`](https://huggingface.co/datasets/triton7777/cube_to_bowl_so100) and [`qb1t/so101_teleop_cubes`](https://huggingface.co/datasets/qb1t/so101_teleop_cubes), each graded by Kalanos 0.10.2 with the same profile, tier and policy.
The notebook charts what each audit decided and the camera evidence behind each review, from the saved reports, so it downloads no dataset.
CPU only, no API key, nothing is published. Choose **Runtime → Run all**.

These are recorded audit decisions, not task-success or model-performance measurements.

## Install

Kalanos and plotly, at pinned versions.

In [ ]:
# @title Install Kalanos
%pip install -q --no-warn-conflicts "kalanos[numeric,hf,video]==0.10.2" plotly==7.1.0

## Load the reports

Each report is the output of one command, run on its pinned source:

```bash
kalanos grade hf://datasets/triton7777/cube_to_bowl_so100@9f490c4e624034eb5c6e961e65767abb9ab7c391 --profile vision-imitation-v1 --tier full --compact --report compare_cube_to_bowl_so100.json
kalanos grade hf://datasets/qb1t/so101_teleop_cubes@2f932198e8925e60f02ed86171f6929845651b92 --profile vision-imitation-v1 --tier full --compact --report compare_so101_teleop_cubes.json
```

The last section runs it again.

In [ ]:
# @title Download and load the reports
from urllib.request import urlretrieve

import kalanos


# The commit in the URL pins the files, so every run shows the same reports.
REPORTS_URL = "https://raw.githubusercontent.com/KalanosAI/kalanos/517777d168b4bf59d56234bfd612f2ed9dab77af/notebooks/reports"
FILES = {
    "triton7777/cube_to_bowl_so100": "compare_cube_to_bowl_so100.json",
    "qb1t/so101_teleop_cubes": "compare_so101_teleop_cubes.json",
}

reports: dict[str, kalanos.Report] = {}
for name, file in FILES.items():
    # Raises on a failed download, so a file left by an earlier run is never read.
    urlretrieve(f"{REPORTS_URL}/{file}", file)
    report = kalanos.load_report(file)
    # `load_report` also reads reports from older schemas, which have no episodes.
    assert isinstance(report, kalanos.Report)
    reports[name] = report

## Scope and coverage

Both audits ran the same Kalanos version, profile, tier and policy, so their decisions compare directly.
Coverage counts, per check, the units each audit computed out of those it needed; "not required" means the profile doesn't ask for that check.

In [ ]:
# @title Show scope and coverage
import polars as pl
from IPython.display import display


def short_name(name: str) -> str:
    """Return a dataset name without its owner, e.g. `cube_to_bowl_so100`."""
    return name.rsplit("/", 1)[-1]


scope_rows = []
for name, report in reports.items():
    counts, source, scope, producer = (
        report.eligibility_counts,
        report.source,
        report.scope,
        report.producer,
    )
    if counts is None or source is None or scope is None or producer is None:
        raise ValueError(f"The report has no decision counts, source or scope: {name}")
    # Sanity checks: the tables and charts must never disagree with the report they are drawn from.
    if not counts.inventory_complete:
        raise ValueError(f"The report has no complete episode inventory: {name}")
    if (
        counts.pass_count + counts.review + counts.blocked + counts.unknown
        != counts.total
    ):
        raise ValueError(f"Decision counts do not sum to the report inventory: {name}")
    scope_rows.append(
        {
            "dataset": name,
            "revision": (source.revision or "")[:7],
            "kalanos": producer.version,
            "profile": scope.requirements_id,
            "tier": scope.tier.value,
            "policy": scope.policy_id,
            "episodes": counts.total,
            "pass": counts.pass_count,
            "review": counts.review,
            "blocked": counts.blocked,
            "unknown": counts.unknown,
        }
    )

# One row per check; `behavioral_diversity` is both an episode check and an evidence dimension.
coverage_rows = {}
for kind, rows_of in (
    ("episode check", lambda coverage: coverage.capabilities),
    ("evidence", lambda coverage: coverage.dimensions),
):
    for name, report in reports.items():
        for row in rows_of(report.coverage):
            cells = coverage_rows.setdefault(
                (kind, row.key), {"check": row.key, "kind": kind, "unit": row.unit}
            )
            cells[short_name(name)] = (
                "not required"
                if row.eligible == 0 and row.not_required > 0
                else f"{row.computed} of {row.eligible}"
            )

display(pl.DataFrame(scope_rows))
display(pl.DataFrame(list(coverage_rows.values())))

## Episode decisions

Each bar splits a dataset's episodes by Kalanos decision.

In [ ]:
# @title Chart episode decisions
import plotly.graph_objects as go


def percent(count: int, total: int) -> str:
    """Return a share as a whole percentage that never rounds a non-zero count to 0% or 100%."""
    share = 100 * count / total
    if 0 < share < 1:
        return "<1%"
    if 99 < share < 100:
        return ">99%"
    return f"{share:.0f}%"


# `pass_` because `pass` is a Python keyword.
statuses = ("pass_", "review", "blocked", "unknown")
labels = ("Pass", "Review", "Blocked", "Unknown")

# All four clear 4.5:1 on the paper. Under red-green colour blindness pass looks like
# unknown and review like blocked, so blocked and unknown also carry a pattern.
STATUS_COLORS = dict(
    zip(statuses, ("#23735e", "#87511b", "#a62b23", "#65625c"), strict=True)
)
STATUS_PATTERNS = {"pass_": "", "review": "", "blocked": "x", "unknown": "."}
PAPER, SURFACE, RULE, INK, MUTED = "#f4f1ea", "#ffffff", "#d2cec5", "#171717", "#65625c"
MONO = "'IBM Plex Mono', monospace"
FONT = {
    "family": "'IBM Plex Sans', Arial, sans-serif",
    "size": 16,
    "color": INK,
}
# Title and subtitle start at the plot's left edge, at the top of the figure.
TITLE = {
    "x": 0,
    "xref": "paper",
    "xanchor": "left",
    "y": 1,
    "yref": "container",
    "yanchor": "top",
    "pad": {"t": 24},
}
OUTCOMES = {"review": "sent to review", "blocked": "blocked", "unknown": "unknown"}
ROW_PX, BAR = 112, 0.6

counts, totals = {}, {}
for name, report in reports.items():
    c = report.eligibility_counts
    # Every report has counts by now; this only tells the type checker.
    assert c is not None
    counts[short_name(name)] = dict(
        pass_=c.pass_count, review=c.review, blocked=c.blocked, unknown=c.unknown
    )
    totals[short_name(name)] = c.total
names = list(counts)

fig = go.Figure()
for key, status_label in zip(statuses, labels, strict=True):
    status_counts = [counts[name][key] for name in names]
    if not any(status_counts):
        continue
    fig.add_bar(
        name=status_label,
        y=names,
        x=[
            100 * c / totals[name] for c, name in zip(status_counts, names, strict=True)
        ],
        text=[
            percent(c, totals[name]) if c else ""
            for c, name in zip(status_counts, names, strict=True)
        ],
        textposition="inside",
        insidetextanchor="middle",
        # Every status colour is dark enough for white text.
        textfont={"color": SURFACE, "size": 16},
        customdata=[
            [c, totals[name]] for c, name in zip(status_counts, names, strict=True)
        ],
        orientation="h",
        width=BAR,
        # A paper-coloured edge leaves a gap between segments.
        marker={
            "color": STATUS_COLORS[key],
            "line": {"color": PAPER, "width": 2},
            "pattern": {
                "shape": STATUS_PATTERNS[key],
                "fgcolor": PAPER,
                "fgopacity": 0.6,
                "size": 10,
            },
        },
        hovertemplate=f"{status_label}: %{{customdata[0]:,}} of %{{customdata[1]:,}} episodes<extra>%{{y}}</extra>",
    )

for name in names:
    # The dataset's name above its bar, so long names leave the bar its full width.
    fig.add_annotation(
        x=0,
        y=name,
        text=f"<b>{name}</b>",
        showarrow=False,
        xanchor="left",
        yanchor="bottom",
        yshift=BAR * ROW_PX / 2 + 4,
    )
    # The not-passed count, at the end of the bar.
    kinds = [key for key in OUTCOMES if counts[name][key]]
    outcome = OUTCOMES[kinds[0]] if len(kinds) == 1 else "did not pass"
    fig.add_annotation(
        x=100,
        y=name,
        text=f"<b>{totals[name] - counts[name]['pass_']:,}</b> of {totals[name]:,}<br>{outcome}",
        showarrow=False,
        xanchor="left",
        align="left",
        xshift=14,
    )

rates = ", ".join(
    f"{name} {percent(counts[name]['review'], totals[name])}" for name in names
)
fig.update_layout(
    font=FONT,
    paper_bgcolor=PAPER,
    plot_bgcolor=PAPER,
    barmode="stack",
    height=200 + ROW_PX * len(names),
    margin={"t": 140, "r": 150, "l": 24, "b": 60},
    title={
        **TITLE,
        "text": f"Review rate: {rates}",
        "font": {"size": 24},
        "subtitle": {
            "text": "Same profile and policy, so the rates compare. The next chart shows why each episode went to review.",
            "font": {"color": MUTED, "size": 16},
        },
    },
    # Percentages too wide for their segment are hidden; hover still gives the count.
    uniformtext={"minsize": 13, "mode": "hide"},
    legend={
        "orientation": "h",
        "x": 0,
        "xanchor": "left",
        "yref": "container",
        "y": 0.02,
        "yanchor": "bottom",
        "traceorder": "normal",
    },
)
fig.update_xaxes(
    range=[0, 100], showticklabels=False, showgrid=False, zeroline=False, showline=False
)
# Datasets top to bottom in the table's order.
fig.update_yaxes(autorange="reversed", showticklabels=False, showline=False, ticks="")
fig.show()

## What sent episodes to review

Every review here is a camera finding: an episode whose video differs from the same camera's other episodes.
The chart puts every episode of that camera on one scale, with the camera's median and the range Kalanos treats as normal.
Blur and brightness are different measurements, so each dataset gets its own axis.

In [ ]:
# @title Chart the camera evidence
import math
import re

from plotly.subplots import make_subplots


# Where each camera finding's per-episode value lives in a graded stream.
EPISODE_VALUE = {
    "vision.blur_vs_camera": ("sharpness_score", "laplacian_var"),
    "vision.exposure_vs_camera": ("exposure_level", None),
}
# Kalanos sets the blur fences on log variance, so blur is drawn on a log axis.
AXES = {
    "vision.blur_vs_camera": (
        "blur, Laplacian variance (log scale): lower is blurrier",
        True,
    ),
    "vision.exposure_vs_camera": ("brightness, mean gray level (0–255)", False),
}
DECISIONS = {
    "pass": ("Pass", STATUS_COLORS["pass_"], "circle"),
    "review": ("Review", STATUS_COLORS["review"], "diamond"),
    "blocked": ("Blocked", STATUS_COLORS["blocked"], "x"),
    "unknown": ("Unknown", STATUS_COLORS["unknown"], "square"),
}
# Neighbours in value sort cycle through five heights, so adjacent values sit on different lines.
JITTER = (-0.5, 0, 0.5, -0.25, 0.25)
# The y range leaves room under the dots for the callout.
Y_RANGE = (-1.8, 1)
PANEL_PX, PANEL_GAP_PX = 260, 130


def episode_label(episode_id: str) -> str:
    """Return `Episode 8` for an id ending in `episode_000008`, else the id after its dataset."""
    match = re.search(r"episode_(\d+)$", episode_id)
    return f"Episode {int(match[1])}" if match else episode_id.rsplit("::", 1)[-1]


def episode_value(
    episode: kalanos.GradedEpisode, metric_id: str, source_field: str
) -> float | None:
    """Return an episode's value for a camera finding's metric, on the finding's camera."""
    metric, key = EPISODE_VALUE[metric_id]
    for stream in episode.streams:
        if stream.source_field == source_field and metric in stream.metrics:
            result = stream.metrics[metric]
            return result.value if key is None else result.evidence.get(key)
    return None


def axis_x(x: float, log: bool) -> float:
    """Return an x position in axis units: on a log axis, plotly places annotations and ranges by log10, and shapes by the data value."""
    return math.log10(x) if log else x


# Findings that changed a decision, one panel per dataset, metric and camera.
flagged = [
    (name, f)
    for name, report in reports.items()
    for f in report.findings
    if f.consequence.value in ("review", "block")
]
panels = {}
for name, f in flagged:
    if f.metric_id in EPISODE_VALUE:
        panels.setdefault((name, f.metric_id, f.source_field), []).append(f)

plot_px = PANEL_PX * len(panels) + PANEL_GAP_PX * (len(panels) - 1)
fig = make_subplots(
    rows=len(panels),
    cols=1,
    vertical_spacing=PANEL_GAP_PX / plot_px if len(panels) > 1 else 0,
)
in_legend = set()
for row, ((name, metric, field), group) in enumerate(panels.items(), start=1):
    axis = "" if row == 1 else str(row)
    heading, log = AXES[metric]
    median = group[0].evidence["camera_median"]
    low_fence, high_fence = group[0].evidence["fences"]
    values = sorted(
        (
            (value, episode)
            for episode in reports[name].episodes
            if (value := episode_value(episode, metric, field)) is not None
        ),
        key=lambda pair: pair[0],
    )
    jitter = [JITTER[i % len(JITTER)] for i in range(len(values))]

    # The x range runs 10% past the outermost value or fence, in the axis's own units.
    xs = [axis_x(v, log) for v, _ in values] + [
        axis_x(low_fence, log),
        axis_x(high_fence, log),
    ]
    pad = 0.1 * (max(xs) - min(xs))
    x_range = [min(xs) - pad, max(xs) + pad]

    # The normal range, and the camera median inside it.
    fig.add_shape(
        type="rect",
        xref=f"x{axis}",
        yref=f"y{axis} domain",
        x0=low_fence,
        x1=high_fence,
        y0=0,
        y1=1,
        fillcolor=SURFACE,
        line_width=0,
        layer="below",
    )
    fig.add_annotation(
        text="normal range",
        xref=f"x{axis}",
        yref=f"y{axis} domain",
        x=axis_x(low_fence, log),
        y=1,
        xanchor="left",
        yanchor="top",
        xshift=6,
        yshift=-4,
        showarrow=False,
        font={"color": MUTED, "size": 14},
    )
    fig.add_shape(
        type="line",
        xref=f"x{axis}",
        yref=f"y{axis} domain",
        x0=median,
        x1=median,
        y0=0,
        y1=1,
        line={"color": MUTED, "width": 1},
        layer="below",
    )
    fig.add_annotation(
        text=f"median {median:.3g}",
        xref=f"x{axis}",
        yref=f"y{axis} domain",
        x=axis_x(median, log),
        y=1,
        xanchor="left",
        yanchor="top",
        xshift=4,
        yshift=-4,
        showarrow=False,
        font={"color": MUTED, "size": 14},
    )

    unit = group[0].unit or ""
    for status, (legend, color, symbol) in DECISIONS.items():
        hits = [
            (v, y, episode)
            for (v, episode), y in zip(values, jitter, strict=True)
            if episode.eligibility is not None
            and episode.eligibility.status.value == status
        ]
        if not hits:
            continue
        fig.add_scatter(
            x=[v for v, _, _ in hits],
            y=[y for _, y, _ in hits],
            mode="markers",
            marker={
                "color": color,
                "symbol": symbol,
                "size": 12,
                "line": {"color": PAPER, "width": 2},
            },
            customdata=[episode_label(episode.id) for _, _, episode in hits],
            name=legend,
            showlegend=legend not in in_legend,
            hovertemplate=f"<b>%{{customdata}}</b><br>%{{x:.3g}} {unit}<extra>{legend}</extra>",
            row=row,
            col=1,
        )
        in_legend.add(legend)

    # One callout, on the flagged episode furthest from the median.
    focus = max(
        group,
        key=lambda f: abs(math.log(f.value / median)) if log else abs(f.value - median),
    )
    focus_y = next(
        y
        for (_, episode), y in zip(values, jitter, strict=True)
        if episode.id == focus.episode_id
    )
    if log:
        gap = (
            f"the camera's median is {median / focus.value:.1f}× higher"
            if focus.value < median
            else f"{focus.value / median:.1f}× the camera's median"
        )
    else:
        side = "below" if focus.value < median else "above"
        gap = f"{abs(focus.value - median):.0f} gray levels {side} the camera's median"
    # Point away from the nearer edge so the box stays inside the figure.
    on_right = axis_x(focus.value, log) > (x_range[0] + x_range[1]) / 2
    fig.add_annotation(
        x=axis_x(focus.value, log),
        y=focus_y,
        xref=f"x{axis}",
        yref=f"y{axis}",
        text=f"<b>{episode_label(focus.episode_id)}</b>: {focus.value:.3g}<br>{gap}",
        ax=-120 if on_right else 120,
        # Centred in the space under the dots, converted from y units to pixels.
        ay=(focus_y - (Y_RANGE[0] + 0.5)) * PANEL_PX / (Y_RANGE[1] - Y_RANGE[0]),
        align="left",
        xanchor="right" if on_right else "left",
        standoff=8,
        bgcolor=SURFACE,
        bordercolor=RULE,
        borderpad=8,
        arrowcolor=MUTED,
    )

    fig.update_xaxes(type="log" if log else "linear", range=x_range, row=row, col=1)
    if log:
        # Ticks at 1, 2 and 5 of each decade, labelled in full.
        decades = range(math.floor(x_range[0]), math.ceil(x_range[1]) + 1)
        ticks = [m * 10**d for d in decades for m in (1, 2, 5)]
        fig.update_xaxes(
            tickvals=ticks, ticktext=[f"{t:g}" for t in ticks], row=row, col=1
        )
    fig.update_yaxes(range=Y_RANGE, visible=False, row=row, col=1)
    # Panel heading at the plot's left edge, above this panel's own plot area.
    camera = field.removeprefix("observation.images.")
    fig.add_annotation(
        text=f"<b>{short_name(name)}</b><br>{camera} {heading}",
        align="left",
        xref="paper",
        yref=f"y{axis} domain",
        x=0,
        y=1,
        xanchor="left",
        yanchor="bottom",
        yshift=12,
        showarrow=False,
    )

charted = sum(len({f.episode_id for f in group}) for group in panels.values())
fig.update_xaxes(gridcolor=RULE, linecolor=RULE, ticks="", zeroline=False)
fig.update_layout(
    font=FONT,
    paper_bgcolor=PAPER,
    plot_bgcolor=PAPER,
    height=plot_px + 300,
    margin={"t": 190, "b": 90, "l": 24, "r": 40},
    title={
        **TITLE,
        "text": f"{charted} review episodes, each outside its own camera's normal range",
        "font": {"size": 24},
        "subtitle": {
            "text": "One dot per episode. White band: the range Kalanos treats as normal for that camera.",
            "font": {"color": MUTED, "size": 16},
        },
    },
    legend={
        "orientation": "h",
        "x": 0,
        "xanchor": "left",
        "yref": "container",
        "y": 0.02,
        "yanchor": "bottom",
    },
)
fig.show()

# One evidence line per finding that changed a decision, charted or not.
for name, f in flagged:
    median = f.evidence.get("camera_median")
    against = "" if median is None else f" against a camera median of {median:.2f}"
    print(
        f"{short_name(name)}, {episode_label(f.episode_id)}: {f.metric_id} on {f.source_field}, {f.value:.2f} {f.unit or ''}".rstrip()
        + against
    )

## Turn the comparison into a project decision

- Both audits use the same Kalanos version, profile and policy. The datasets, robots and findings differ: one camera is blurred, the other overexposed, so don't compare blur variance with gray level.
- Choose a source for your hypothesis and write down one validation still needed. Full recorded visual coverage does not establish capture timing or training suitability.
- In a real experiment, pin the dataset revision, record train and validation IDs, fit any normalization on the actual train IDs, and compare models under matched held-out evaluation and budgets.

## Regrade and compare

Optional. Tick `REGRADE` to grade both pinned sources again with the profile and tier each report records, and to check that the counts and review findings match the saved reports.
The charts above keep the saved reports.
It downloads about 480 MB and takes about ten minutes on a CPU runtime.

In [ ]:
# @title Regrade and compare
import logging


REGRADE = False  # @param {type:"boolean"}


def finding_keys(report: kalanos.Report) -> list[tuple]:
    """Return the findings that changed a decision as sorted, comparable tuples."""
    return sorted(
        (f.episode_id, f.metric_id, f.source_field or "", round(f.value, 6))
        for f in report.findings
        if f.consequence.value in ("review", "block")
    )


if not REGRADE:
    print("Regrade off. Tick REGRADE to grade both datasets again.")
else:
    # As the CLI's `--verbosity error`: hides warnings about the optional adapters this install leaves out.
    logging.getLogger("kalanos").setLevel(logging.ERROR)
    for name, saved in reports.items():
        # Every report has a scope and source by now; this only tells the type checker.
        assert saved.scope is not None and saved.source is not None
        profile, tier = saved.scope.requirements_id, saved.scope.tier
        print(f"Grading {name} with {profile} / {tier.value}…", flush=True)
        # The profile name is the bundle the CLI's `--profile` passes to `grade`.
        fresh = kalanos.grade(saved.source.uri, bundle=profile, tier=tier)
        differences = [
            label
            for label, same in (
                ("counts", fresh.eligibility_counts == saved.eligibility_counts),
                ("findings", finding_keys(fresh) == finding_keys(saved)),
            )
            if not same
        ]
        verdict = (
            "same counts and findings as the saved report"
            if not differences
            else f"{' and '.join(differences)} differ from the saved report"
        )
        version = fresh.producer.version if fresh.producer else "unknown"
        print(f"{name}: Kalanos {version}, {verdict}\n")

**Continue with Kalanos:** [Browse dataset reports](https://hub.kalanos.ai) · [Use the grader locally](https://github.com/KalanosAI/kalanos) · [Attach an audit to a training run](https://github.com/KalanosAI/kalanos-tracking)